In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 데이터 로드 및 전처리
# ==========================================
# sklearn에서 iris 데이터 로드 (샘플 150개, 특성 4개, 클래스 3개)
iris = load_iris()
X_raw = iris.data      # shape: (150, 4) - 꽃받침 길이/너비, 꽃잎 길이/너비
y_raw = iris.target    # shape: (150,)   - 0, 1, 2 세 개 클래스 (이미 인덱스 형태라 원-핫 불필요)

# 학습용/테스트용 분리 (8:2 비율, stratify로 클래스 비율 유지)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 표준화(정규화): 평균 0, 표준편차 1로 스케일 맞춤
# -> 특성마다 값의 범위가 다르면(예: cm 단위 차이) 학습이 불안정해질 수 있어서 필요
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)  # train 데이터로 평균/표준편차 계산 후 적용
X_test_scaled = scaler.transform(X_test_raw)         # test에는 train에서 구한 값을 그대로 적용 (누수 방지)

# numpy 배열 -> PyTorch 텐서로 변환
X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train = torch.tensor(y_train_raw, dtype=torch.long)  # CrossEntropyLoss는 클래스 인덱스(long) 요구
y_test = torch.tensor(y_test_raw, dtype=torch.long)

# ==========================================
# 2. 신경망 모델 정의 (기존 MultiClassNet 그대로 재사용)
# ==========================================
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 CrossEntropyLoss 내부에서 처리
        return out

# ==========================================
# 3. 모델, 손실 함수, 옵티마이저 생성
# ==========================================
input_size = X_train.shape[1]   # iris 특성 개수 = 4 (하드코딩 대신 데이터에서 자동으로 뽑음)
hidden_size = 8                 # 은닉 노드 개수 (기존 5에서 조금 늘림, 자유롭게 조정 가능)
output_size = 3                 # iris 클래스 개수 (setosa, versicolor, virginica)
learning_rate = 0.1             # 기존 0.5는 iris엔 다소 클 수 있어 낮춤

torch.manual_seed(42)
model = MultiClassNet(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# ==========================================
# 4. 학습 루프
# ==========================================
print("=== PyTorch 학습 시작 (Iris) ===")
epochs = 3000
for epoch in range(epochs):
    outputs = model(X_train)          # 순전파
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y_train).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Train Accuracy: {accuracy.item():.1f}%")

# ==========================================
# 5. 테스트 데이터로 최종 성능 확인
# ==========================================
print("\n=== 테스트 데이터 예측 ===")
model.eval()
with torch.no_grad():
    logits = model(X_test)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)
    test_accuracy = (predictions == y_test).float().mean() * 100

print(f"테스트 정확도: {test_accuracy.item():.1f}%")
for i in range(5):  # 앞 5개 샘플만 확인
    prob_list = [round(p, 3) for p in probabilities[i].tolist()]
    actual = iris.target_names[y_test[i].item()]
    pred = iris.target_names[predictions[i].item()]
    print(f"샘플 {i+1} 확률분포: {prob_list} -> 예측: {pred} (실제: {actual})")

=== PyTorch 학습 시작 (Iris) ===
Epoch  500 | Loss: 0.2863 | Train Accuracy: 93.3%
Epoch 1000 | Loss: 0.1633 | Train Accuracy: 96.7%
Epoch 1500 | Loss: 0.1109 | Train Accuracy: 97.5%
Epoch 2000 | Loss: 0.0867 | Train Accuracy: 97.5%
Epoch 2500 | Loss: 0.0737 | Train Accuracy: 97.5%
Epoch 3000 | Loss: 0.0658 | Train Accuracy: 97.5%

=== 테스트 데이터 예측 ===
테스트 정확도: 96.7%
샘플 1 확률분포: [0.987, 0.013, 0.0] -> 예측: setosa (실제: setosa)
샘플 2 확률분포: [0.0, 0.238, 0.762] -> 예측: virginica (실제: virginica)
샘플 3 확률분포: [0.042, 0.958, 0.0] -> 예측: versicolor (실제: versicolor)
샘플 4 확률분포: [0.022, 0.978, 0.0] -> 예측: versicolor (실제: versicolor)
샘플 5 확률분포: [0.991, 0.009, 0.0] -> 예측: setosa (실제: setosa)
